# 01. Exploratory Data Analysis & Modeling Formulation
**Project:** Solar Power Generation Prediction & Performance Deviation Analysis  
**Role:** Role 1 (Machine Learning Engineer)  
**Dataset:** Kaggle `anikannal/solar-power-generation-data` (Plants 1 & 2, 22 inverters each, 15-min intervals)  
**Contracts & Specs:** `docs/DATA_AND_ML_SPEC.md` (sections 1-7), `docs/ARCHITECTURE_AND_API.md` (section 3.5), `docs/DECISION_LOG.md` (D-004, D-006, D-007, D-015)

## 1. Executive Summary & Raw Data Quirks
- **Dataset Span:** ~34 calendar days (May 15, 2020 to June 17, 2020) across both plants.
- **Inverters:** Exactly 22 distinct inverters in Plant 1 and 22 distinct inverters in Plant 2.
- **Date Parsing Quirk:** Plant 1 Generation file uses day-first format (`%d-%m-%Y %H:%M`); all other files (Plant 1 Weather, Plant 2 Generation, Plant 2 Weather) use ISO format (`%Y-%m-%d %H:%M:%S`). Explicit parsing prevents catastrophic month-day transposition.
- **DC/AC Power Scale:** Plant 1 raw `DC_POWER` is approximately 10.2x `AC_POWER` (due to inverter unit scaling); Plant 2 raw `DC_POWER` is approximately 1.02x `AC_POWER`.
- **Peak Irradiance:** Daily peak irradiance ranges from 1.10 to 1.25 kW/m², confirming values are in kW/m².
- **Leakage Guard:** Fleet-normal model uses weather and time features only (`irradiation_kw_m2`, temperatures, cyclical hour encodings, trailing weather features). No inverter ID, DC power, yield meters, or lagged AC power are used as features.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Load generated data report
report_path = Path('../artifacts/processed/data_report.json')
if report_path.exists():
    with open(report_path, 'r') as f:
        data_report = json.load(f)
    print(json.dumps(data_report, indent=2))
else:
    print('Data report artifact not found.')

## 2. Quality Flags & Data Cleaning Integrity
Per `docs/DATA_AND_ML_SPEC.md` section 4, six deterministic quality flags are computed:
1. `flag_missing_weather`: Row has no corresponding weather reading.
2. `flag_interpolated_weather`: Weather gap <= 2 intervals was linearly interpolated.
3. `flag_zero_ac_positive_dc`: Daylight row where DC > 0 but AC == 0.
4. `flag_positive_ac_zero_dc`: Row where AC > 0 but DC == 0.
5. `flag_extreme_dc_ac_ratio`: DC/AC ratio outside [8.0, 14.0] for Plant 1 or [0.7, 1.4] for Plant 2.
6. `flag_flatlined_ac`: Consecutive identical daylight readings over >= 8 intervals (2 hours).
7. `flag_any`: True if any of the above flags is raised.

**Result:** 98.63% of rows in Plant 1 and 94.80% in Plant 2 are completely clean (`flag_any == False`).

## 3. Robust Training Analysis (Spec Section 7 & Decision D-015)
### Hypothesis & Rule Formulation
In solar PV plants, persistently underperforming inverters (e.g., due to persistent soiling, string disconnection, or sensor drift) can contaminate training data. If included, they drag down the expected fleet-normal power curve.

**Robust Training Filter Rule:**
Drop training rows where the inverter's peer ratio (`inverter_ac / fleet_median_ac`) is `< 0.70` or `> 1.30` at timestamps where the fleet median is at least 5% of capacity proxy.

### Empirical Benchmark: Robust Training ON vs OFF

In [2]:
comparison_path = Path('../artifacts/metrics/robust_training_comparison.csv')
if comparison_path.exists():
    comp_df = pd.read_csv(comparison_path)
    display_cols = ['plant_id', 'robust', 'model', 'train_rows', 'val_daylight_rmse', 'val_daylight_mae', 'test_daylight_rmse', 'test_daylight_mae', 'test_daylight_wape']
    print(comp_df[display_cols].to_string(index=False))
else:
    print('Robust training comparison artifact not found.')

### Findings & Decision Rationale
- **Plant 1:** The inverter fleet is highly uniform. Only 124 outlier rows (0.5%) are dropped. Validation and test metrics are virtually identical between ON and OFF (Physics Baseline test RMSE is 53.91 kW OFF vs 53.96 kW ON).
- **Plant 2:** Several inverters suffer from chronic, persistent underperformance. Robust training correctly identifies and drops 972 outlier rows (4.7%).
  - **Physics Baseline:** Test Daylight MAE drops from **92.48 kW to 84.66 kW** (an 8.5% improvement); WAPE improves from **20.51% to 18.77%**.
  - **Polynomial Ridge:** Test Daylight MAE drops from **100.67 kW to 81.30 kW** (a **19.2% reduction in error!**); WAPE improves from **22.32% to 18.03%**.
  - **Hist Gradient Boosting:** Test Daylight MAE drops from **93.69 kW to 81.92 kW** (a **12.6% reduction in error!**); WAPE improves from **20.78% to 18.17%**.

**Conclusion:** Robust training successfully prevents contaminated inverters from dragging down the baseline expectation, improving accuracy across all models on clean, normal operation. **Robust training is ACCEPTED for production (Decision D-015).**

## 4. Model Benchmarks & Parsimony Selection (D-006)
Models are evaluated chronologically on the unseen validation block (2020-06-07 to 2020-06-11):
1. `physics_baseline`: $P = a \cdot G \cdot (1 + \gamma (T_{\text{mod}} - 25))$
2. `ridge_poly`: Degree-2 polynomial with Ridge regularization
3. `hist_gradient_boosting`: HistGradientBoosting with monotonic constraint on irradiance
4. `random_forest`: 100 trees with max depth 12

**3% Parsimony Rule:** If a simpler model's validation daylight RMSE is within 3% of the best complex model, the simpler model is selected.
- **Plant 1:** Best complex model is `hist_gradient_boosting` (110.19 kW). `physics_baseline` achieves 111.97 kW (within 1.6% of best). Simpler physics model is selected!
- **Plant 2:** `physics_baseline` achieves the lowest validation daylight RMSE (207.15 kW) directly.

## 5. Permutation Feature Importance & Physical Explanations
Permutation importance is evaluated on validation daylight rows (5 repeats, seed 42):

| Feature | Normalized Importance | Physical & Engineering Explanation |
|---|---|---|
| `irradiation_kw_m2` | 96.4% | Direct solar irradiance driving photovoltaic excitation and electron flow. Primary physical energy input. |
| `module_temp_c` | 3.6% | PV panel surface temperature. Higher temperatures increase semiconductor resistance and decrease open-circuit voltage (~ -0.3% to -0.4% per °C above 25°C). |
| `ambient_temp_c` | < 0.1% | Outdoor air temperature. Governs convective heat transfer from panel surface to atmosphere. |
| `temp_delta_c` | < 0.1% | Gradient between module and ambient air (`module_temp_c - ambient_temp_c`), proxy for convective cooling. |
| `hour_sin` / `hour_cos` | < 0.1% | Cyclical diurnal solar angle encodings. Irradiance already captures elevation, so residual importance is minimal. |
| `irr_lag1` / `irr_diff1` | < 0.1% | Short-term temporal irradiance momentum and cloud transit ramping. |
| `irr_roll_mean_4` / `std` | < 0.1% | 1-hour trailing average and volatility. Captures cloud intermittency. |
| `irr_x_modtemp` | < 0.1% | Non-linear interaction between incident irradiance and thermal loss. |

In [3]:
metrics_path = Path('../artifacts/metrics/model_metrics.json')
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        m = json.load(f)
    for p_id in ['1', '2']:
        p_data = m['plants'][p_id]
        print(f"\nPlant {p_id} Selected Model: {p_data['selected_model']}")
        print(f"Selection Reason: {p_data['selection_reason']}")
        print(f"Capacity Proxy: {p_data['capacity_proxy_kw']} kW")
        print("Top Features:")
        for feat in p_data.get('feature_importance', [])[:4]:
            print(f"  - {feat['feature']}: {feat['importance']*100:.1f}%")
else:
    print('Model metrics artifact not found.')